In [9]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model

In [14]:
IMAGE_DIR = "LabelStudioToMask_FINAL/images"
MASK_DIR = "LabelStudioToMask_FINAL/masks"
SAVE_DIR = "LabelStudioToMask_FINAL/predictions"     # saved in parent folder
os.makedirs(SAVE_DIR, exist_ok=True)
print("1. Paths loaded")

1. Paths loaded


In [15]:
IMG_SIZE = 256

def load_image(image_path):
    image = tf.io.decode_png(tf.io.read_file(image_path), channels=1)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image

def load_mask(mask_path):
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = tf.cast(mask, tf.float32) / 255.0
    return mask

def load_pair(image_path):
    # Extract filename safely
    parts = tf.strings.split(image_path, '/')
    image_name = parts[-1]                      # filename: RVG_1.png
    
    # Remove .png extension
    name_no_ext = tf.strings.regex_replace(image_name, ".png", "")
    
    # Build mask filename: RVG_1_mask.png
    mask_name = tf.strings.join([name_no_ext, "_mask.png"])
    mask_path = tf.strings.join([MASK_DIR, "/", mask_name])

    return load_image(image_path), load_mask(mask_path)

image_files = sorted([IMAGE_DIR + "/" + f for f in os.listdir(IMAGE_DIR)])
dataset = tf.data.Dataset.from_tensor_slices(image_files)
dataset = dataset.map(load_pair).shuffle(200).batch(4).prefetch(2)

# Split for train/val/test
val_data = dataset.take(1)
test_data = dataset.skip(1).take(1)
train_data = dataset.skip(2)

print("2. Dataset loaded")

2. Dataset loaded


In [16]:
def attention_block(x, g, filters):
    theta_x = layers.Conv2D(filters, (2, 2), strides=(2, 2), padding='same')(x)
    phi_g = layers.Conv2D(filters, (1, 1), padding='same')(g)
    add = layers.add([theta_x, phi_g])
    act = layers.Activation('relu')(add)
    psi = layers.Conv2D(1, (1, 1), padding='same')(act)
    psi = layers.Activation('sigmoid')(psi)
    psi = layers.UpSampling2D(size=(2, 2))(psi)
    return layers.multiply([x, psi])
print("3. Attention block loaded.")

3. Attention block loaded.


In [ ]:
def attention_unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D()(c3)

    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = layers.Conv2D(1024, 3, activation='relu', padding='same')(p4)
    bn = layers.Conv2D(1024, 3, activation='relu', padding='same')(bn)

    # Decoder + Attention
    g4 = layers.Conv2D(512, 1)(bn)
    att4 = attention_block(c4, g4, 512)
    u4 = layers.UpSampling2D()(bn)
    u4 = layers.concatenate([u4, att4])
    c5 = layers.Conv2D(512, 3, activation='relu', padding='same')(u4)
    c5 = layers.Conv2D(512, 3, activation='relu', padding='same')(c5)

    g3 = layers.Conv2D(256, 1)(c5)
    att3 = attention_block(c3, g3, 256)
    u3 = layers.UpSampling2D()(c5)
    u3 = layers.concatenate([u3, att3])
    c6 = layers.Conv2D(256, 3, activation='relu', padding='same')(u3)
    c6 = layers.Conv2D(256, 3, activation='relu', padding='same')(c6)

    g2 = layers.Conv2D(128, 1)(c6)
    att2 = attention_block(c2, g2, 128)
    u2 = layers.UpSampling2D()(c6)
    u2 = layers.concatenate([u2, att2])
    c7 = layers.Conv2D(128, 3, activation='relu', padding='same')(u2)
    c7 = layers.Conv2D(128, 3, activation='relu', padding='same')(c7)

    g1 = layers.Conv2D(64, 1)(c7)
    att1 = attention_block(c1, g1, 64)
    u1 = layers.UpSampling2D()(c7)
    u1 = layers.concatenate([u1, att1])
    c8 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)
    c8 = layers.Conv2D(64, 3, activation='relu', padding='same')(c8)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c8)

    return Model(inputs, outputs)

model = attention_unet()
print("4. Model loaded.")

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.MeanIoU(num_classes=2)]
)
print("5. Model compiled.")

model.summary()

4. Model loaded.
5. Model compiled.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 256, 256, 1)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_35 (Conv2D)            │ (None, 256, 256, 64)      │             640 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_36 (Conv2D)            │ (None, 256, 256, 64)      │          36,928 │ conv2d_35[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_4               │ (None, 128, 128, 64)      │               0 │ conv2d_36[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_37 (Conv2D)            │ (None, 128, 128, 128)     │          73,856 │ max_pooling2d_4[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_38 (Conv2D)            │ (None, 128, 128, 128)     │         147,584 │ conv2d_37[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_5               │ (None, 64, 64, 128)       │               0 │ conv2d_38[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_39 (Conv2D)            │ (None, 64, 64, 256)       │         295,168 │ max_pooling2d_5[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_40 (Conv2D)            │ (None, 64, 64, 256)       │         590,080 │ conv2d_39[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_6               │ (None, 32, 32, 256)       │               0 │ conv2d_40[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_41 (Conv2D)            │ (None, 32, 32, 512)       │       1,180,160 │ max_pooling2d_6[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_42 (Conv2D)            │ (None, 32, 32, 512)       │       2,359,808 │ conv2d_41[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_7               │ (None, 16, 16, 512)       │               0 │ conv2d_42[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_43 (Conv2D)            │ (None, 16, 16, 1024)      │       4,719,616 │ max_pooling2d_7[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_44 (Conv2D)            │ (None, 16, 16, 1024)      │       9,438,208 │ conv2d_43[0][0]            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 33,818,757 (129.01 MB)

 Trainable params: 33,818,757 (129.01 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("6. Training model...\n")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25
)
print("7. Model trained.")

6. Training model...

Epoch 1/25


In [ ]:
print("8. Loss curves")
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train Loss', 'Val Loss'])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.show()

In [ ]:
print("9. Testing the model.")
for i, (img, mask) in enumerate(test_data.take(10)):
    pred = model.predict(img)
    pred = (pred > 0.5).astype(np.uint8) * 255

    for j in range(pred.shape[0]):
        save_path = os.path.join(SAVE_DIR, f"pred_{i}_{j}.png")
        tf.keras.preprocessing.image.save_img(save_path, pred[j])
